# RoadLens on Colab (T4 GPU)

Traffic sign classification robust to image corruptions: training, the six experiments and all figures.

**How to use**
1. Upload **`RoadLens.zip`** and this notebook to the top level of **My Drive**.
2. `Runtime > Change runtime type > T4 GPU`.
3. `Runtime > Run all`. Approve the Google Drive prompt when it appears.

**What happens**
- The first cell unpacks the code to `MyDrive/RoadLens/` and the cached datasets to the local Colab disk.
- Checkpoints (`runs*/`) and results (`results*/`, CSV + LaTeX tables + PDF figures) are written to `MyDrive/RoadLens/`, so they survive the session.
- The corrupted test sets (GTSRB-C) are generated in the background while the first models train.

**If the session ends early**: open the notebook again and `Run all`. Finished training runs and evaluations are skipped; a training run that was cut off starts again from its first epoch (per-epoch checkpoints are kept on the local disk, not on Drive, so they do not fill your Drive).

This notebook starts with the **pilot** scale (1 seed, shortened schedules, about 1-1.5 hours). It has not been run on Colab before, so treat the pilot as the first real test. When it finishes cleanly, set `PROFILE = ''` in the second code cell and `Run all` again for the full three-seed experiments.

In [ ]:
import os, subprocess, sys, time, zipfile
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
ZIP = f'{DRIVE}/RoadLens.zip'        # the bundle you uploaded
UPDATE = f'{DRIVE}/RoadLens_update.zip'   # optional: a newer copy of the code alone (scripts/make_bundle.py --update)
PROJECT = f'{DRIVE}/RoadLens'        # code, checkpoints (runs*/) and results (results*/) live here
DATA = '/content/roadlens_data'      # datasets on the fast local disk; rebuilt in every session

def zip_version(path):
    with zipfile.ZipFile(path) as z:
        return z.read('RoadLens/VERSION').decode().strip()

installed = open(f'{PROJECT}/VERSION').read().strip() if os.path.exists(f'{PROJECT}/VERSION') else None
for src in sorted((p for p in (ZIP, UPDATE) if os.path.exists(p)), key=zip_version):   # oldest first
    if installed is None or zip_version(src) > installed:      # first run, or newer code was uploaded
        with zipfile.ZipFile(src) as z:
            z.extractall(DRIVE, [m for m in z.namelist() if m.startswith('RoadLens/')])
        installed = zip_version(src)
        print(f'code installed from {os.path.basename(src)} (version {installed}) to {PROJECT}')
if os.path.exists(ZIP) and not os.path.isdir(f'{DATA}/processed'):
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('/content', [m for m in z.namelist() if m.startswith('roadlens_data/')])
    print('cached datasets unpacked to', DATA)
assert os.path.isdir(f'{PROJECT}/roadlens'), f'Upload RoadLens.zip to the top level of My Drive first (looked for {ZIP})'

%cd {PROJECT}
os.environ['ROADLENS_DATA'] = DATA
os.environ['ROADLENS_SCRATCH'] = '/content/roadlens_scratch'   # per-epoch checkpoints stay off Drive
!pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then run this cell again'
print('GPU:', torch.cuda.get_device_name(0), f'({torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB)')

## Scale

| `PROFILE` | what it does | GPU time on a T4 (rough estimate, not measured) | outputs |
|---|---|---|---|
| `'smoke'` | every script on tiny subsets, 2 epochs: only checks that the pipeline runs; numbers are meaningless | ~15 min | `runs_smoke/`, `results_smoke/` |
| `'pilot'` | full training set, **1 seed**, schedules shortened to 40 %, first 2,000 test images: real but rough numbers | ~1-1.5 h | `runs_pilot/`, `results_pilot/` |
| `''` | the full experiments: **seeds 0, 1, 2**, full schedules, full test sets | ~10-12 h (two or three sessions) | `runs/`, `results/` |

The profiles never overwrite each other. To get a measured estimate for this GPU, set `RUN_CHECKS = True` in the optional cell at the end.

In [ ]:
PROFILE = 'pilot'          # 'smoke' | 'pilot' | ''  ('' = the full three-seed experiments)

P = f'--profile {PROFILE}' if PROFILE else ''
RESULTS = 'results' + (f'_{PROFILE}' if PROFILE else '')
print('profile:', PROFILE or 'full', '| results ->', f'{PROJECT}/{RESULTS}')

## Data

The GTSRB and BTSD caches (64 px) come from the bundle; if they are missing they are downloaded instead.
The corrupted test sets (14 corruptions x 5 severities) are generated in the background on one CPU core
while training starts: about 45 minutes for the full sets, a few minutes for the pilot subset.

In [ ]:
!python scripts/prepare_data.py --datasets gtsrb btsd

CACHE = f'{DRIVE}/RoadLens_cache'            # optional copy of the corrupted sets on Drive (see the last cells)
if os.path.isdir(CACHE):
    !rsync -a "{CACHE}/" "{DATA}/"

if 'gen' in globals() and gen.poll() is None:
    print('the corrupted test sets are already being generated')
else:
    cmd = (f'{sys.executable} scripts/make_gtsrb_c.py --workers 1 {P} && '
           f'{sys.executable} scripts/make_gtsrb_c.py --dataset btsd --workers 1 {P}')
    gen = subprocess.Popen(cmd, shell=True, stdout=open('/content/make_c.log', 'w'), stderr=subprocess.STDOUT)
    print('generating the corrupted test sets in the background (log: /content/make_c.log)')

def wait_for_corrupted_sets():
    while gen.poll() is None:
        lines = open('/content/make_c.log').read().replace('\r', '\n').strip().splitlines()
        print(time.strftime('%H:%M:%S'), 'still generating:', lines[-1][-100:] if lines else '')
        time.sleep(120)
    assert gen.returncode == 0, open('/content/make_c.log').read()[-3000:]
    print('corrupted test sets are ready')

from IPython.display import Image, display
if os.path.exists('results/corruptions/grid_0.png'):
    display(Image('results/corruptions/grid_0.png', width=420))    # what the 14 x 5 corruptions look like

## Experiments

Each cell trains whatever it still needs (shared models are trained once and reused) and writes CSV, text and LaTeX tables to `results*/<experiment>/`.

In [ ]:
# Experiment 1 - clean accuracy / macro-F1 of all models. Trains SmallCNN, ResNet-18, DeiT-S (baseline and
# proposed training) and the gated fusion. Does not need the corrupted test sets.
!python scripts/exp1_clean.py {P}

In [ ]:
# Experiment 2 - every corruption x severity: accuracy, macro-F1, mCE, relative mCE, ECE (main table)
wait_for_corrupted_sets()
!python scripts/exp2_corruption.py {P}

In [ ]:
# Experiment 3 - ablation: baseline -> +mixing -> +JSD -> +Fourier/PRIME ops -> +fusion -> +TENT
!python scripts/exp3_ablation.py {P}

In [ ]:
# Experiment 4 - leave-category-out selection of the augmentation op set (trains two more ResNet-18 variants)
!python scripts/exp4_leave_category_out.py {P}

In [ ]:
# Experiment 5 - cross-dataset: BTSD (CTSD is skipped until its class map is verified, see README)
!python scripts/exp5_cross_dataset.py {P}

In [ ]:
# Experiment 6 - test-time BN / TENT vs. test batch size (1, 16, 64, 200)
!python scripts/exp6_tent_batchsize.py {P}

In [ ]:
# Experiment 7 - the v2 approach, one mechanism per row: per-image input standardisation, view experts,
# the stacked degradation-aware gate, temperature scaling; and test-time BN with a source-statistics prior.
# Trains two branches and one gate (about 20 minutes at pilot scale) and reuses everything above.
wait_for_corrupted_sets()
!python scripts/exp7_improved.py {P}

In [ ]:
# Figures (vector PDF): severity curves, confusion matrices, reliability diagrams, Fourier heat map, Grad-CAM
!python scripts/make_figures.py {P}

import glob
for path in ['exp1_clean/clean.txt', 'exp2_corruption/main.txt', 'exp2_corruption/per_corruption.txt',
             'exp3_ablation/ablation.txt', 'exp4_leave_category_out/candidates.txt',
             'exp4_leave_category_out/selection.txt', 'exp5_cross_dataset/btsd.txt', 'exp6_tent_batchsize/sweep.txt',
             'exp7_improved/improved.txt', 'exp7_improved/severity5.txt', 'exp7_improved/adaptation.txt']:
    full = os.path.join(RESULTS, path)
    if os.path.exists(full):
        print(f'\n===== {path} =====')
        print(open(full).read())
print('\nfigures:')
for f in sorted(glob.glob(f'{RESULTS}/**/*.pdf', recursive=True)):
    print('  ', f)
print(f'\nEverything is in {PROJECT}/{RESULTS} on your Drive.')

In [ ]:
import glob
for path in ['exp1_clean/clean.txt', 'exp2_corruption/main.txt', 'exp2_corruption/per_corruption.txt',
             'exp3_ablation/ablation.txt', 'exp4_leave_category_out/candidates.txt',
             'exp4_leave_category_out/selection.txt', 'exp5_cross_dataset/btsd.txt', 'exp6_tent_batchsize/sweep.txt']:
    full = os.path.join(RESULTS, path)
    if os.path.exists(full):
        print(f'\n===== {path} =====')
        print(open(full).read())
print('\nfigures:')
for f in sorted(glob.glob(f'{RESULTS}/**/*.pdf', recursive=True)):
    print('  ', f)
print(f'\nEverything is in {PROJECT}/{RESULTS} on your Drive.')

In [ ]:
# Preview the main PDF figures inline
try:
    !pip install -q pymupdf
    import fitz
    for pdf in ['figures/severity_curves.pdf', 'figures/reliability.pdf', 'figures/fourier_heatmap.pdf',
                'figures/gradcam.pdf', 'exp6_tent_batchsize/tent_batchsize.pdf']:
        full = os.path.join(RESULTS, pdf)
        if os.path.exists(full):
            print(pdf)
            display(Image(data=fitz.open(full)[0].get_pixmap(dpi=110).tobytes('png')))
except Exception as err:
    print('preview skipped:', err)

## Optional

- `RUN_CHECKS`: run the unit tests and measure this GPU's speed; prints an estimated GPU time per experiment.
- `SAVE_C_TO_DRIVE`: copy the generated corrupted test sets to `MyDrive/RoadLens_cache/` so that later sessions
  restore them in a few minutes instead of regenerating them. Needs free Drive space: about 13.3 GB for the
  full sets, 3.4 GB for the pilot subsets.

In [ ]:
RUN_CHECKS = False
SAVE_C_TO_DRIVE = False

if RUN_CHECKS:
    !python -m pytest tests -q
    !python scripts/benchmark.py
if SAVE_C_TO_DRIVE:
    wait_for_corrupted_sets()
    !mkdir -p "{CACHE}" && rsync -a --include='*_c_s*/***' --exclude='*' "{DATA}/" "{CACHE}/"
    !du -sh "{CACHE}"